In [1]:
%pip install dotenv
%pip install datasets
%pip install scikit-learn
%pip install -r requirements.txt
%load_ext autoreload
%autoreload 2


  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dotenv]
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import sys

sys.path.append(".")


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [4]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CPU cores:", os.cpu_count())


CPU cores: 8


In [27]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()

# Use your token to log in
login(token=os.getenv("hf_token"))


In [ ]:
print(f"Number of rows: {len(df)}")
print(f"Number of unique group_uids: {len(df['group_uid'].unique())}")
print(f"Average number of articles per group_uid: {len(df) / len(df['group_uid'].unique()):.2f}")


In [ ]:
import sys

sys.path.append("/utils")

from utils.experiment_utils import run_experiment, DatasetConfig, ExperimentConfig


In [ ]:
def run_custom(model, loss):
    # Model Selection (choose one)
    MODEL = f"{model}/epoch-1.pt"

    # Output directory
    OUTPUT_DIR = f"{model}/"

    # Dataset Configuration
    DATASET_CONFIG = DatasetConfig(
        custom_dataset="upasanachatterjee/article-bias-prediction-media-splits-updated",
        theme=None,
        k_means=None,
        trunc=True,
        sentiment=False,
        no_undersampling=True,
        media_split=True,
    )

    # Experiment Configuration
    EXPERIMENT_CONFIG = ExperimentConfig(
        loss_type=loss,  # Options: "standard", "focal", "weighted_asymmetric_focal"
        focal_alpha=1.0,
        focal_gamma=2.0,
        gamma_pos=1.0,
        gamma_neg=4.0,
        patience=3,
        save_model=False,
    )

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Run the experiment
    print("Starting experiment...")
    print(f"Model: {MODEL}")
    print(f"Loss type: {EXPERIMENT_CONFIG.loss_type}")
    print(f"Output directory: {OUTPUT_DIR}")
    print("-" * 50)

    try:
        results = run_experiment(
            model=MODEL,
            loc=OUTPUT_DIR,
            dataset_config=DATASET_CONFIG,
            experiment_config=EXPERIMENT_CONFIG,
        )
        print("Experiment completed successfully!")
        print(f"Results saved to: {OUTPUT_DIR}")

        if isinstance(results, dict) and len(results) > 1:
            print(f"Multiple theme results: {list(results.keys())}")

    except Exception as e:
        print(f"Experiment failed with error: {str(e)}")
        raise e


In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
import glob
from utils.experiment_utils import (
    run_experiment, DatasetConfig, ExperimentConfig,
    ALLSIDES_EXTENDED_MEDIA_SPLIT, ALLSIDES_EXTENDED_RANDOM_SPLIT,
)
from utils.model_utils import BERT, BART, ROBERTA, POLITICS, IDEOLOGY_CLASSIFIER


In [ ]:
ideology_dir = snapshot_download(repo_id=IDEOLOGY_CLASSIFIER)
ideology_pt = glob.glob(f"{ideology_dir}/*.pt")[0]
print(f"Ideology classifier .pt: {ideology_pt}")


In [ ]:
MODELS = [BERT, BART, ROBERTA, POLITICS, ideology_pt]

def run_all_models(dataset, output_prefix, no_undersampling=False):
    os.makedirs(output_prefix, exist_ok=True)
    results = {}
    for model in MODELS:
        cfg = DatasetConfig(custom_dataset=dataset, no_undersampling=no_undersampling)
        exp = ExperimentConfig(patience=3, num_epochs=15, save_model=False)
        print(f"\n{'='*60}")
        print(f"Model: {model}  |  Dataset: {dataset}  |  no_undersampling={no_undersampling}")
        print('='*60)
        val_metrics, test_metrics = run_experiment(
            model=model,
            loc=output_prefix,
            dataset_config=cfg,
            experiment_config=exp,
        )
        results[str(model)] = test_metrics
    return results


In [ ]:
media_results_undersampling = run_all_models(
    dataset=ALLSIDES_EXTENDED_MEDIA_SPLIT,
    output_prefix="results_undersampling/media_split",
    no_undersampling=False,
)


In [ ]:
random_results_undersampling = run_all_models(
    dataset=ALLSIDES_EXTENDED_RANDOM_SPLIT,
    output_prefix="results_undersampling/random_split",
    no_undersampling=False,
)


In [ ]:
media_results_no_undersampling = run_all_models(
    dataset=ALLSIDES_EXTENDED_MEDIA_SPLIT,
    output_prefix="results_no_undersampling/media_split",
    no_undersampling=True,
)


In [ ]:
random_results_no_undersampling = run_all_models(
    dataset=ALLSIDES_EXTENDED_RANDOM_SPLIT,
    output_prefix="results_no_undersampling/random_split",
    no_undersampling=True,
)
